# Demo: ledger2 find

How do you find things in the ledger without knowing their names?
Treat it as a table whose rows are subjects and whose columns are
predicates: `refs` returns the subjects matching a where clause, as
`Ref`s; `select` returns chosen columns of them, as tuples. This
notebook fills a ledger with runs, versions and identity-model memo
entries, then asks questions of it.

In [1]:
import os, shutil, sys, tempfile
from pathlib import Path

import pandas as pd

sys.path.insert(0, "demo_svp")          # the stand-in svp package
from dvc_dat.ledger2 import Code, Env, Ledger, Unindexed
from dvc_dat.ledger2 import set_env_provider
from svp.shots import Shots

ENV = Env({"sv-pipe": Code("0.2.2", "7d742989")},
          lock="sha256:5e1f", platform="darwin-arm64-py3.11")
old_provider = set_env_provider(lambda module: ENV)
ledger_dir = Path(tempfile.mkdtemp(prefix="demo-l2-find-"))
os.environ["DAT_LEDGER"] = str(ledger_dir)
L = Ledger.open()
with L.transaction(by="human:dan") as t:
    t.add("code/sv-pipe", "version", "0.2.2")
    t.add("code/sv-pipe", "commit", "7d742989")
    t.add("model/player-id", "version", "2.1.0")
    t.add("model/shot", "version", "0.2.2")
    for g in ("G1", "G4", "G5"):
        frames = pd.DataFrame({"frame": range(int(g[1:]))})
        t.add(f"run/0-2-2/{g}", "value", Shots(frames))
    t.add("run/0-2-1/G5", "value", Shots(pd.DataFrame({"f": [0]})))
    t.add("ledger/pred/result.acc", "index", "true")
    t.add("ledger/pred/result.loss", "index", "true")

rev = L.at()
for game, epochs in (("G1", 8), ("G4", 8), ("G5", 8), ("G5", 12)):
    rev.eval("svp.identity.train", game=game, epochs=epochs)
print(L.at())

Rev(5)


Four runs, three versioned subjects, four memo entries. Two dotted
predicates are declared indexed: `result.acc` and `result.loss`.

## `refs`: which subjects

In [2]:
rev = L.at()
print(rev.refs(s="run/0-2-2/*"))
print(rev.refs(s="run/0-2-1/G5"))
print([r.subject for r in rev.refs(where={"version": None})])
print([r.subject for r in rev.refs(where={"version": "0.2.2"})])

[Ref(5, 'run/0-2-2/G1'), Ref(5, 'run/0-2-2/G4'), Ref(5, 'run/0-2-2/G5')]
[Ref(5, 'run/0-2-1/G5')]
['code/sv-pipe', 'model/player-id', 'model/shot']
['code/sv-pipe', 'model/shot']


`s=` is an exact subject or a prefix ending in `*`. `where` maps a
predicate to a value, or to `None` for "any value".

In [3]:
entries = rev.refs(fn="svp.identity.train", game="G5")
print(entries)
e = entries[-1]
print(dict(e.rev.all(e.subject))["kwargs.epochs"], e.load().obj)

[Ref(5, 'Pickled/1ffae442f3e9444b'), Ref(5, 'Pickled/bce827b525c175b6')]
12 {'model': 'identity-G5-12', 'base': '2.1.0', 'frames': 5, 'acc': 0.93}


A bare keyword is a key of a memo entry (`kwargs.<k>`), and `fn=` is
its function. A `Ref` is one subject at one revision: its rows are
readable without loading, and `load()` fetches the value.

## `select`: which values

In [4]:
for row in rev.select(("game", "epochs", "result.acc"),
                      fn="svp.identity.train"):
    print(row)

('G1', '8', '0.88')
('G4', '8', '0.88')
('G5', '8', '0.93')
('G5', '12', '0.93')


`keys` names the columns, in order; each match comes back as a
tuple. `game` and `epochs` are keys; `result.acc` is the indexed
attribute `seal` computed from the result. Values come back as text.

In [5]:
print(rev.select(("game", "epochs"), fn="svp.identity.train",
                 where={"result.acc": "0.93"}))
print(rev.select(("version",), s="model/*"))

[('G5', '8'), ('G5', '12')]
[('2.1.0',), ('0.2.2',)]


Keys, `where` and `s=` combine: the G5 entries by accuracy, and the
version of every `model/` subject.

## An index declared later

In [6]:
with L.transaction(by="human:dan") as t:
    t.add("ledger/pred/result.frames", "index", "true")
rev = L.at()
rev.eval("svp.identity.train", game="G4", epochs=12)
print(rev.select(("game", "epochs"), fn="svp.identity.train",
                 where={"result.frames": None}))
print(L.at().select(("game", "epochs", "result.frames"),
                    fn="svp.identity.train",
                    where={"result.frames": None}))

[]
[('G4', '12', '4')]


A declaration covers the seals made after it; earlier entries do
not get the row. `select` reads as of its revision, so the new G4
entry shows only at the top.

## `one=`

In [7]:
rev = L.at()
print(rev.refs(fn="svp.identity.train", game="G1", one=True))
for game in ("G5", "G7"):
    try:
        rev.refs(fn="svp.identity.train", game=game, one=True)
    except LookupError as e:
        print(type(e).__name__, "-", e)

Ref(7, 'Pickled/fdef44a915fd7846')
Multiple - refs: 2 subjects matched
KeyError - 'refs: no subject matched'


`one=True` returns the single match itself, and raises `Multiple`
for several and `KeyError` for none.

## `Unindexed`

In [8]:
print(rev.refs(where={"result.loss": "0.1"}))
try:
    rev.refs(where={"result.accuracy": "0.93"})
except Unindexed as e:
    print("Unindexed:", e)

[]
Unindexed: 'result.accuracy' is not declared indexed (ledger/pred/result.accuracy index true) and no row has it at rev 7


A dotted predicate that is declared but has no rows answers `[]`. One
neither declared nor present on any row (here a typo) raises.

In [9]:
set_env_provider(old_provider)
shutil.rmtree(ledger_dir)
del os.environ["DAT_LEDGER"]
print("removed:", not ledger_dir.exists())

removed: True
